# Chroma DB Filtering

**ChromaDB filtering** = narrowing down query results using conditions on **metadata** (e.g., `category="tech"`) or **document content** (e.g., contains a certain word), applied *alongside* similarity search — so you get results that are both semantically similar *and* match specific criteria.

Filtering in **Chroma DB** fundamentally differs from traditional SQL-based filtering due to its emphasis on **vector similarity** and flexible **metadata querying**. While SQL databases rely on structured schemas and declarative logic to retrieve exact matches, Chroma DB is designed for **unstructured data** and **semantic search**, making it well-suited for AI-driven applications.

## 2 Primary Types Of ChromaDB Filtering

| Filter Type | Description | Comparison to SQL |
|---|---|---|
| **Metadata Filtering** | Filters based on document metadata, such as `"topic": "history"` or `"date": "2023-01-15"` | Similar to SQL `WHERE` clauses, but more flexible and can be combined with vector search |
| **Document Filtering** | Filters based on document content using keyword presence (e.g., `$contains`, `$not_contains`) | Comparable to SQL's `CONTAINS` or `LIKE` operators, but more powerful when integrated with vector search |

This dual-filtering approach allows Chroma DB to support complex, context-aware queries that go beyond the capabilities of traditional relational databases.

> 💡 Document filtering in Chroma DB is also referred to as **full text search**.

## Metadata Filtering in Chroma DB

Metadata filtering in Chroma DB can be performed by using the `where` parameter inside the `.query()`, `.get()`, or `.delete()` methods.

For basic metadata matches, where you want to find only equivalent matches, use the following syntax:

```python
where={"key": "value"}
```

For instance, the following code will `get` only those documents within `collection` where the metadata `key` is exactly equal to `value` :

```python
collection.get(
    where={"key": "value"}
)
```

Similar syntax can be used within `.query()` or `.delete()` methods.

## ChromaDB — Metadata Filtering Operators

### What These Are

Beyond simple `"key": "value"` matching, ChromaDB supports **comparison operators** for metadata filters — same idea as SQL comparison operators (`=`, `!=`, `>`, `<`, etc.), just written differently.

### The Operators

| Operator | Meaning | Works on |
|---|---|---|
| `$eq` | equal to | string, int, float |
| `$ne` | not equal to | string, int, float |
| `$gt` | greater than | int, float |
| `$gte` | greater than or equal to | int, float |
| `$lt` | less than | int, float |
| `$lte` | less than or equal to | int, float |

### Syntax

```python
where={"key": {"$eq": "value"}}
```

Read this as: *"filter where `key`'s value equals `value`"*

**Example with a number:**
```python
where={"year": {"$gt": 2020}}
```
→ only return documents where `year > 2020`

## The Shortcut

```python
where={"key": {"$eq": "value"}}
```
is **exactly the same** as:
```python
where={"key": "value"}
```

**Why:** if you don't specify an operator, ChromaDB assumes you mean "equal to" by default. So the short form `{"key": "value"}` is just a convenient shorthand for `{"key": {"$eq": "value"}}`.

### ChromaDB — Combining Filters (`$and` / `$or`)

Combine multiple conditions using `$and` / `$or` logical operators.

```python
collection.get(
    where={
        "$and": [
            {"key": {"$eq": "value1"}},
            {"key": {"$ne": "value2"}}
        ]
    }
)
```

This gets documents where `key == value1` **AND** `key != value2`.

- `$or` works the same way — swap `$and` for `$or` to match if *either* condition is true
- Same syntax works with `.get()`, `.query()`, and `.delete()`

### ChromaDB — List Filters (`$in` / `$nin`)

Use `$in` / `$nin` to filter against a **list of values**.

- `$in` → matches if `key`'s value is **in** the given list
- `$nin` → matches if `key`'s value is **not in** the given list ("not in")

```python
where={"key": {"$nin": ["value1", "value2"]}}
```

This finds all documents where `key` is **not equal to either** `value1` or `value2`.

Note that,

```python
where={"key": {"$nin": ["value1", "value2"]}}
```

is just a **shorthand** for:

```python
where={
    "$and": [
        {"key": {"$ne": "value1"}},
        {"key": {"$ne": "value2"}}
    ]
}
```

Both mean: *"give me documents where `key` is not `value1` AND not `value2`."* `$nin` is just cleaner to write, especially as your exclude-list grows longer — imagine excluding 5 values with `$and`/`$ne` chains vs one clean `$nin` list.

### ChromaDB — Document Filtering

Filters based on the actual **text content** of documents (not metadata) — checks if certain text/keywords are present.

Applied via the `where_document` parameter, in `.query()`, `.get()`, or `.delete()`.

```python
where_document={"$contains": "value"}
```

Finds all documents whose text **contains** `"value"`.

- `$not_contains` → opposite, excludes documents containing that text
- Can combine multiple document filters with `$and` / `$or`, same syntax pattern as metadata filters

**Key distinction to remember:** `where` = filters on metadata, `where_document` = filters on the actual document text.

## A full example of metadata and document filtering in Chroma DB

## 🔧 Setup

The following commands import `chromadb` and its **embedding utilities**, then create an embedding function object used to generate **vector embeddings**:

In [2]:
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(    # SentenceTransformerEmbeddingFunction is a wrapper class provided by ChromaDB that tells Chroma: "use the sentence-transformers library to generate embeddings, specifically using this model."
    model_name="all-MiniLM-L6-v2"    # embedding model name - This embedding model is small, fast, popular sentence-transformer model from Hugging Face, commonly used as a default because it's lightweight
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 📂 Creating Collections

A **collection** in ChromaDB is the equivalent of a table in a traditional database — it's a named container where you store your documents, their embeddings, and associated metadata together. All your data lives inside collections, and you can have multiple collections for different purposes (e.g., one for product descriptions, one for support docs). Every add/query/delete operation you do in ChromaDB is always done on a specific collection.

Note that 1 collection is **not** just for one document.

#### The hierarchy

```
ChromaDB (the database)
    └── Collection (like a table)
            └── Documents (many — could be thousands)
```

#### What each level is

**ChromaDB itself** = the database. The whole thing.

**Collection** = a named group/container *inside* that database. Think of it like a folder or a table. One collection holds **many documents** that are related to each other in some way.

**Documents** = the actual individual pieces of text/data stored inside a collection.

#### Concrete example

Say you're building a RAG system for a company. You might create:

- Collection `"product_docs"` → stores 500 product description documents
- Collection `"support_tickets"` → stores 10,000 customer support messages
- Collection `"internal_policies"` → stores 50 HR policy documents

Each collection holds many documents. You keep them separate so when you query, you search within the right group — searching `"product_docs"` won't accidentally return support tickets.

**So one collection = one "topic/category" of many documents**

### Create a Collection

**Firstly let's dive into Clients :-**

`chromadb.Client()` creates a **ChromaDB client object** — it's your entry point to interact with ChromaDB. Think of it as the "connection" to the database, same way you'd create a database connection object in any other DB library. Everything you do (create collections, add data, query) goes through this `client` object.

In **standalone mode** (which is what `chromadb.Client()` creates), there's no separate server at all — the client *is* everything. It's just a Python object that manages the database entirely within your current process/script. No HTTP, no separate server process, nothing external.

Think of it like this: `chromadb.Client()` spins up a tiny in-memory database instance right inside your Python script, and the `client` object is how you talk to it.

The "client-server" distinction only matters when you deploy ChromaDB in production — where you'd run a separate ChromaDB server process, and *then* your Python code connects to it over HTTP as a true client. In that case `client` really is a network client sending requests to a remote server.

But for learning/local work (which is what you're doing now), `chromadb.Client()` = standalone mode = no server, just an in-memory DB object living inside your script.

**For a local RAG chatbot (personal project, portfolio)**

You'd still use standalone mode, but with **persistent storage** so data survives between runs:

```python
client = chromadb.PersistentClient(path="./chroma_db")
```

This saves your vectors to disk locally instead of in-memory — no server needed, just a folder on your machine. This is what most personal RAG projects use.

**For a production app (deployed, multiple users hitting it)**

Then yes, you'd run an actual ChromaDB server separately, and your app connects to it as a client over HTTP:

```python
client = chromadb.HttpClient(host="localhost", port=8000)
```

But this is only relevant when you're deploying something real with real users — not for learning or portfolio projects.

#### So the realistic progression for you

1. **Right now (learning):** `chromadb.Client()` — in-memory, no persistence
2. **Building a RAG portfolio project:** `chromadb.PersistentClient()` — saves to disk, still no server
3. **Actual production deployment someday:** `chromadb.HttpClient()` — real client-server setup

For interview projects and portfolio work, `PersistentClient` is all you'll ever need.

In [3]:
client = chromadb.Client()

collection = client.create_collection(
    name="filter_demo",
    metadata={"description": "Used to demo filtering in ChromaDB"},
    configuration={    # Just a dictionary where you pass settings for the collection.
        "embedding_function": ef
    }
)

print(f"Collection created: {collection.name}")

Collection created: filter_demo


## ➕ Adding Documents to Collections

Use `add` to insert documents with optional metadata.

In [6]:
collection.add(
    documents=[
        "This is a document about LangChain",
        "This is a reading about LlamaIndex",
        "This is a book about Python",
        "This is a document about pandas",
        "This is another document about LangChain"
    ],
    metadatas=[
        {"source": "langchain.com", "version": 0.1},
        {"source": "llamaindex.ai", "version": 0.2},
        {"source": "python.org", "version": 0.3},
        {"source": "pandas.pydata.org", "version": 0.4},
        {"source": "langchain.com", "version": 0.5},
    ],
    ids=["id1", "id2", "id3", "id4", "id5"]
)

**What if there is length mismatch in these?**

it'll give an error. ChromaDB strictly requires **documents, metadatas, and ids to all have the same length** — they map to each other by position (index 0 of documents pairs with index 0 of metadatas and index 0 of ids, and so on).

If the counts don't match, ChromaDB will throw a `ValueError` before anything gets stored.

Also worth knowing — **ids must be unique** within a collection. If you try to add a document with an id that already exists in that collection, ChromaDB will raise an error for that too.

## 🏷️ Filter using Metadata

The following finds all documents where the source is **"langchain.com"**:

In [7]:
collection.get(
    where={"source": {"$eq": "langchain.com"}}
)

{'ids': ['id1', 'id5'],
 'embeddings': None,
 'documents': ['This is a document about LangChain',
  'This is another document about LangChain'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'langchain.com', 'version': 0.1},
  {'source': 'langchain.com', 'version': 0.5}]}

The above produced correct output, but suppose we were only interested in LangChain documents with versions less than 0.3. The following finds all documents where the source is "langchain.com" with versions less than 0.3:

In [8]:
collection.get(
    where={
        "$and": [
            {"source": {"$eq": "langchain.com"}}, 
            {"version": {"$lt": 0.3}}
        ]
    }
)

{'ids': ['id1'],
 'embeddings': None,
 'documents': ['This is a document about LangChain'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'version': 0.1, 'source': 'langchain.com'}]}

Now, let's make an even more complicated filtering rule, one that combines logical operators with lists in filters. The following retrieves all documents about LangChain and LlamaIndex with a version less than 0.3:

In [10]:
collection.get(
    where={
        "$and": [
            {"source": {"$in": ["langchain.com", "llamaindex.ai"]}}, 
            {"version": {"$lt": 0.3}}
        ]
    }
)

{'ids': ['id1', 'id2'],
 'embeddings': None,
 'documents': ['This is a document about LangChain',
  'This is a reading about LlamaIndex'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'langchain.com', 'version': 0.1},
  {'source': 'llamaindex.ai', 'version': 0.2}]}

## 📝 Filter using Document Content

Suppose we wanted to find documents that include the word "pandas" in the text. The following performs a full text search for such documents:

In [11]:
collection.get(
    where_document={"$contains":"pandas"}
)

{'ids': ['id4'],
 'embeddings': None,
 'documents': ['This is a document about pandas'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'source': 'pandas.pydata.org', 'version': 0.4}]}

> 💡 Document filtering is case-sensitive in Chroma DB. Therefore, searching for "Pandas" will not find any documents.

## 🏷️ + 📝 Combine Metadata and Document Content Filters

Of course, we can combine metadata and document filters. The following looks for all documents containing "LangChain" or "Python" with version numbers greater than 0.1:

In [13]:
collection.get(
    where={"version": {"$gt": 0.1}},
    where_document={
        "$or": [
            {"$contains": "LangChain"},
            {"$contains": "Python"}
        ]
    }
)

{'ids': ['id3', 'id5'],
 'embeddings': None,
 'documents': ['This is a book about Python',
  'This is another document about LangChain'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'version': 0.3, 'source': 'python.org'},
  {'source': 'langchain.com', 'version': 0.5}]}

### ✅ Wrap-up

In this notebook, you explored how to implement Chroma DB's robust metadata and document filtering features to selectively include or exclude documents based on a predefined set of criteria. Metadata filtering allows you to target documents using structured attributes like tags, topics, or timestamps, while document filtering enables you to search within the content itself—either through keyword matching or semantic relevance. Together, these filtering techniques empower you to build more precise, context-aware queries that enhance the quality and relevance of your search results in AI-driven applications.